


# Phase 3 — Frequency-Aware Multi-Band EEG Representation

An interactive educational walkthrough of **Phase 3: Frequency-Aware EEG Representation**. This notebook demonstrates multi-band spectral decomposition (Theta, Alpha, Beta, Gamma) using zero-phase FIR filtering, tensor transformations, YAML configuration loading, and debug artifact exports.


## 2. Objective

Standard time-domain EEG signals blend multiple oscillatory sources into a single voltage trace per electrode.

- **What Phase 3 Does**: Decomposes single-band time signals into multi-band spectral tensors using zero-phase FIR bandpass filters via MNE (`mne.filter.filter_data`).
- **Why It Exists**: Motor Imagery (MI) tasks induce frequency-specific power variations (Event-Related Desynchronization / Synchronization). Isolating frequency sub-bands enables downstream attention networks to learn spectral band importance.
- **Problem Solved**: Transforms 2D single-band window tensors `(Channels, Samples)` into 3D multi-band tensors `(Bands, Channels, Samples)`, and 3D window batches `(N, Channels, Samples)` into 4D spectral tensors `(N, Bands, Channels, Samples)`.


## 3. Pipeline Position

```text
Time-Domain Epochs (N_epochs, Channels, Samples)
                     ↓
FrequencyRepresentation.extract()  [zero-phase FIR filtering]
                     ↓
Multi-Band Epoch Tensor (N_epochs, Bands, Channels, Samples)
                     ↓
Sliding Window Segmentation
                     ↓
PyTorch HGDDataset (representation="frequency") -> [Bands, Channels, Samples] = [4, 133, 250]
```


## 4. Neurophysiological Theory & Sub-Bands

Motor Imagery execution modulates power across four primary neurophysiological frequency bands:

1. **Theta (4.0 - 8.0 Hz)**: Frontal midline synchronization associated with task initiation, attention allocation, and cognitive control.
2. **Alpha (8.0 - 13.0 Hz)**: Sensorimotor Mu rhythm Event-Related Desynchronization (ERD) over contralateral motor cortex during motor mental imagery.
3. **Beta (13.0 - 30.0 Hz)**: Desynchronization during active imagery, followed by post-imagery Event-Related Synchronization (ERS / beta rebound).
4. **Gamma (30.0 - 38.0 Hz)**: High-frequency local motor network synchronization involved in fine motor control representation.

### Zero-Phase FIR Filtering:
Forward-backward zero-phase FIR filtering (`phase="zero"`, `firwin` design) eliminates phase distortion, ensuring all frequency sub-bands remain perfectly phase-aligned across time.


## 5. Demonstration: Extracting Multi-Band Spectral Tensors

We import `FrequencyRepresentation`, `EEGPreprocessingPipeline`, and `HGDDataset`.


In [ ]:
import os
import sys

def get_project_root():
    curr = os.path.abspath(os.getcwd())
    while curr and not os.path.exists(os.path.join(curr, "datasets")):
        parent = os.path.dirname(curr)
        if parent == curr:
            break
        curr = parent
    return curr

PROJECT_ROOT = get_project_root()
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(f"[OK] Project Root set to: {PROJECT_ROOT}")
import numpy as np
import matplotlib.pyplot as plt
from scipy.fft import fft, fftfreq

from datasets.transforms.frequency import (
    FrequencyBandConfig,
    FrequencyRepresentationConfig,
    FrequencyRepresentation,
)
from datasets.pipeline import EEGPreprocessingPipeline
from datasets.dataset import HGDDataset

print("[OK] Successfully imported FrequencyRepresentation modules!")


### Single Window Transformation Test

We test `FrequencyRepresentation.extract()` on a single synthetic window of shape `(133, 250)`.


In [ ]:
# Create synthetic single window: (Channels, Samples) = (133, 250)
np.random.seed(42)
dummy_window = np.random.randn(133, 250).astype(np.float32)

freq_rep = FrequencyRepresentation()
out_tensor, metadata = freq_rep.extract(dummy_window)

print("=" * 60)
print("SINGLE WINDOW TRANSFORMATION SUMMARY")
print("=" * 60)
print(f"Input Window Shape : {dummy_window.shape} (Channels x Samples)")
print(f"Output Tensor Shape: {out_tensor.shape} (Bands x Channels x Samples)")
print(f"Sub-band Order     : {[b['name'] for b in metadata.frequency_bands]}")
print(f"Execution Duration : {metadata.execution_time_seconds:.4f} seconds")
print("=" * 60)


## 6. Visualizations: Multi-Band Signal Decomposition & FFT Spectral Power

We visualize the 4-subband waveform decomposition and Fast Fourier Transform (FFT) power spectra for a sample EEG window.


In [ ]:
# 1. 4-Subband Waveform Decomposition for Channel 1
if 'out_tensor' in locals():
    time_win = np.linspace(0, 1.0, 250)
    band_names = ["Theta (4-8 Hz)", "Alpha (8-13 Hz)", "Beta (13-30 Hz)", "Gamma (30-38 Hz)"]
    colors = ["#756bb1", "#2b5c8f", "#d95f02", "#31a354"]

    fig, axes = plt.subplots(4, 1, figsize=(12, 8), sharex=True)
    fig.suptitle("Multi-Band Spectral Decomposition (Single Window, Channel 1)", fontsize=13, fontweight="bold")

    for b_idx in range(4):
        axes[b_idx].plot(time_win, out_tensor[b_idx, 0, :], color=colors[b_idx], linewidth=1.2)
        axes[b_idx].set_ylabel(f"{band_names[b_idx]}", fontsize=9, fontweight="bold")
        axes[b_idx].grid(True, linestyle="--", alpha=0.5)

    axes[-1].set_xlabel("Time (seconds)", fontsize=10, fontweight="bold")
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()


In [ ]:
# 2. Fast Fourier Transform (FFT) Power Spectral Density across Sub-Bands
if 'out_tensor' in locals():
    fs = 250.0
    N = 250
    xf = fftfreq(N, 1 / fs)[:N // 2]

    fig, ax = plt.subplots(figsize=(10, 4.5))

    for b_idx in range(4):
        yf = fft(out_tensor[b_idx, 0, :])
        psd = 2.0 / N * np.abs(yf[0:N // 2])
        ax.plot(xf, psd, label=band_names[b_idx], color=colors[b_idx], linewidth=1.5)

    ax.set_title("FFT Power Spectral Density (PSD) per Frequency Sub-Band", fontsize=12, fontweight="bold")
    ax.set_xlabel("Frequency (Hz)", fontsize=10, fontweight="bold")
    ax.set_ylabel("Spectral Amplitude", fontsize=10, fontweight="bold")
    ax.set_xlim(0, 50)
    ax.grid(True, linestyle="--", alpha=0.5)
    ax.legend(frameon=True, facecolor="#F8F9FA")

    plt.tight_layout()
    plt.show()


## 7. Pipeline & Dataset Mode Comparison Results

Comparing PyTorch dataset tensors under `representation="time"` vs `representation="frequency"`.


In [ ]:
sample_edf = os.path.join(PROJECT_ROOT, "hgd", "train1", "1.edf")

if os.path.exists(sample_edf):
    pipeline = EEGPreprocessingPipeline()

    # Time Representation Dataset
    ds_time = HGDDataset(sample_edf, pipeline=pipeline, representation="time")
    sample_time, _ = ds_time[0]

    # Frequency Representation Dataset
    ds_freq = HGDDataset(sample_edf, pipeline=pipeline, representation="frequency")
    sample_freq, _ = ds_freq[0]

    print("=" * 65)
    print("           REPRESENTATION MODE COMPARISON SUMMARY")
    print("=" * 65)
    print(f"Time Mode PyTorch Sample Shape     : {sample_time.shape} (Channels x Samples)")
    print(f"Frequency Mode PyTorch Sample Shape: {sample_freq.shape} (Bands x Channels x Samples)")
    print(f"Sub-band Channels Preserved        : {sample_freq.shape[1]} channels across {sample_freq.shape[0]} bands")
    print(f"Frequency Metadata                 : {ds_freq.metadata}")
    print("=" * 65)


## 8. Conclusion

### Key Accomplishments in Phase 3:
1. **Multi-Band Decomposition**: Implemented zero-phase FIR sub-band filtering (Theta, Alpha, Beta, Gamma).
2. **Tensor Shape Transformations**: Expanded 2D/3D matrices into 3D/4D multi-band tensors seamlessly.
3. **Pipeline & PyTorch Integration**: Supported both `representation="time"` and `representation="frequency"` in `HGDDataset`.
4. **Debug Artifact Export**: Enabled structured debug exports (`frequency_tensor.npy`, `frequency_metadata.json`, `frequency_summary.json`).

### Next Step:
Proceed to **Phase 4 (Adaptive Channel Attention)** to design spatial and spectral channel attention blocks operating on 4D multi-band EEG tensors!
